This file aims to pre-process data to be implemented in MSPA
* recode background and foreground to MSPA
* reproject data if needed
* export as geotiff and byte/8-bit
* uses dask to parallelize processes 

In [1]:
# Import Packages
import os
import re
import glob
from pathlib import Path
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling
from pyproj import CRS

In [2]:
# Initialize Dask cluster
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='20GB')
client = Client(cluster)
print(f"Dashboard: {client.dashboard_link}")

# failed with workers =6, memoery = 8- failed in less than 1 min
# (n_workers=4, threads_per_worker=1, memory_limit='8GB') failed after running almost 4 mins
# n_workers=2, threads_per_worker=2, memory_limit='20GB') - 4 mins and 45 sec, took a couple mins to export raster

Dashboard: http://127.0.0.1:8787/status


In [3]:
cwd = os.getcwd()
print(cwd)

/home/gisuser/code


In [11]:
from pathlib import Path

input_dir = "./data"
for p in Path(input_dir).iterdir():
    if p.is_file():
        print(p)

data/area_etm.etm
data/mosaic.zip
data/typology_TimeInterval.xlsx
data/typology_TS.xlsx
data/~$typology_TimeInterval.xlsx
data/~$typology_TS.xlsx


In [3]:
# Parameters
input_dir =  "./data/DS_TEST/mosaic" #"./code/data/INPUT_FOLDER"
output_dir = "./data/DS_TEST/mosaic_P_recoded" #"./code/data/OUPUT_FOLDER"
tif_pattern = r'mosaic_(\d{4})\.tif'
background1 = 0 # original image background value
foreground1 = 1 # original image foreground value
background2 = 1 # target background value
foreground2 = 2 # target foreground value
source_crs = "EPSG:4326"
#target_crs = "+proj=aea +lat_0=-32 +lon_0=-60 +lat_1=-5 +lat_2=-42 +x_0=0 +y_0=0 +datum=SAD69 +units=m +no_defs"

#"EPSG:102033" #"EPSG:4618"  # Set to None to skip reprojection

# unique projection for the area of interest
from pyproj import CRS
#target_crs = CRS.from_proj4("+proj=aea +lat_0=-32 +lon_0=-60 +lat_1=-5 +lat_2=-42 +x_0=0 +y_0=0 +ellps=aust_SA +towgs84=-66.87,4.37,-38.52,0,0,0,0 +units=m +no_defs +type=crs")
target_crs = "+proj=aea +lat_0=-32 +lon_0=-60 +lat_1=-5 +lat_2=-42 +x_0=0 +y_0=0 +ellps=aust_SA +towgs84=-66.87,4.37,-38.52,0,0,0,0 +units=m +no_defs"

    #"+proj=aea +lat_0=-32 +lon_0=-60 +lat_1=-5 +lat_2=-42 +x_0=0 +y_0=0 +ellps=aust_SA +units=m +no_defs +type=crs")


def process_files(input_dir, output_dir, source_crs, target_crs, background1, background2, foreground1, foreground2, tif_pattern):
    """
    This function processes all TIF files from a folder, recoding the background and foreground to suit MSPA input, reporject if needed,
    and exports image as GEOTIFF in 8-bit which is needed for MSPA. This is optimized to be used with Dask.

    Inputs:
    input_dir : filepath
        The path to the input folder containing the file or files.
    output_dir : filepath
        The path to the output folder where files will be exported.
    source_crs : str
        Contains the original data or "source" CRS that will be used for transforming. The format should be 'EPSG:XXXX'.
    target_crs : str
        Contains the target CRS that will be used for reprojecting. The format should be 'EPSG:XXXX'.
    background1 : int
        Contains the original images background value.
    background2 : int
        Contains the desired background value to be recoded to.
    foreground1 : int
        Contains the original images foreground value.
    foreground2 : int
        Contains the desired foreground value to be recoded to.
    tif_pattern : str
        Pattern of the tif files to extract the year from the file name.
    """
    # Read all TIFF files
    tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
    print(f"Found {len(tif_files)} files to process")
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    results = []
    for tif_file in tif_files:
        try:
            filename = os.path.basename(tif_file)
            print(f"Processing: {filename}")
            
            # Open with Dask chunking for lazy loading
            data = rxr.open_rasterio(
                tif_file,
                chunks={'band': 1, 'x': -1, 'y': 'auto'},
                lock=False #each will have their own file reader instead of being a bottle neck 
            )

            # Prep files to be in xarray if no metadata
            if data.rio.crs is None:
                print(f"CRS missing in {filename}, setting to {source_crs}")
                data = data.rio.write_crs(source_crs)
            # transform 
            # if not data.rio.transform():
            #     print(f"Transform missing in {filename}, setting from file")
            #     with rxr.open_rasterio(tif_file) as src:
            #         data = data.rio.write_transform(src.rio.transform())
            # Calculate transform if missing
            if data.rio.transform() is None:
                from affine import Affine
                x_res = float(data.x[1] - data.x[0])
                y_res = float(data.y[1] - data.y[0])
                transform = Affine.translation(float(data.x[0]) - x_res/2, float(data.y[0]) - y_res/2) * Affine.scale(x_res, y_res)
                data = data.rio.write_transform(transform)
            
            # Recode data using Dask
            print(f"Recoding: {filename}")
            recoded = xr.where(data == foreground1, foreground2,
                      xr.where(data == background1, background2, 0)).astype('uint8')
            
            # 4. Preserve all metadata explicitly                       ##### there has to be a cleaner way to do this
            recoded = recoded.rio.write_crs(data.rio.crs)
            recoded = recoded.rio.write_transform(data.rio.transform())
            recoded = recoded.rio.write_nodata(0) 

            # Reproject if target CRS is provided                       ##### rasterio warp vs rio? 
            if target_crs:
                print(f"Reprojecting: {filename}")
                recoded = recoded.rio.reproject(
                    target_crs,
                    resampling = Resampling.nearest
                )
            
            # Generate output name                                      ##### NEED to change- if not reprojected = no _P_ in filename
            match = re.search(tif_pattern, filename)
            output_name = f"{match.group(1)}_P_recoded.tif" if match else f"{Path(tif_file).stem}_P_recoded.tif"
            output_path = os.path.join(output_dir, output_name)
            
            # Write with Dask
            print(f"Exporting: {filename}")
            recoded.rio.to_raster(
                output_path,
                dtype="uint8",
                compress='LZW',
                tiled=True,
                lock=True
            )
            print(f"Saved: {output_name}")
            results.append(True)

                                                                        ##### add a completed 1 out of 24 type thing
                                            # # Calculate success rate
                                            # success_count = sum(processing_results)
                                            # print(f"\nProcessed {success_count}/{len(processing_results)} files successfully")

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            results.append(False)
    
    return results

In [ ]:
# test from amazon q
# Parameters
input_dir =  "./data/DS_TEST/mosaic"
output_dir = "./data/DS_TEST/mosaic_P_recoded"
tif_pattern = r'mosaic_(\d{4})\.tif'
background1 = 0
foreground1 = 1
background2 = 1
foreground2 = 2
source_crs = "EPSG:4326"
target_crs = "ESRI:102033"
# target_crs = "+proj=aea +lat_0=-32 +lon_0=-60 +lat_1=-5 +lat_2=-42 +x_0=0 +y_0=0 +ellps=aust_SA +towgs84=-66.87,4.37,-38.52,0,0,0,0 +units=m +no_defs"

def point_transform(coor, src_crs, target_crs=5070):
    """
    This function transforms a coordinate (x, y) from a source crs to a target crs. 
    
    Inputs:
        coor : list
        a list or tuple containing the coordinates in x, y format. 
        src_crs : `proj4` CRS
            The source CRS of the coordinate (it should be in a format known to `proj4`)
        target_crs : `proj4` CRS
            The target CRS for transforming the coordinate. Default is `EPSG:5070` which is the CRS of USDA CDL data.
    
    Return:
        transformed_coor : list
            A list with the transformed coordinates in x, y format. 
    """
    
    proj = pyproj.Transformer.from_crs(src_crs, target_crs, always_xy=True)
    projected_coor = proj.transform(coor[0], coor[1])
    transformed_coor = [projected_coor[0], projected_coor[1]]
    
    return transformed_coor

def process_files(input_dir, output_dir, source_crs, target_crs, background1, background2, foreground1, foreground2, tif_pattern):
    """
    This function processes all TIF files from a folder, recoding the background and foreground to suit MSPA input, reporject if needed,
    and exports image as GEOTIFF in 8-bit which is needed for MSPA. This is optimized to be used with Dask.
    """
    tif_files = glob.glob(os.path.join(input_dir, "*.tif"))
    print(f"Found {len(tif_files)} files to process")
    os.makedirs(output_dir, exist_ok=True)
    
    results = []
    for tif_file in tif_files:
        try:
            filename = os.path.basename(tif_file)
            print(f"Processing: {filename}")
            
            data = rxr.open_rasterio(
                tif_file,
                chunks={'band': 1, 'x': -1, 'y': 'auto'},
                lock=False
            )

            if data.rio.crs is None:
                print(f"CRS missing in {filename}, setting to {source_crs}")
                data = data.rio.write_crs(source_crs)
            
            # source transform 
            if data.rio.transform() is None:
                from affine import Affine
                x_res = float(data.x[1] - data.x[0])
                y_res = float(data.y[1] - data.y[0])
                transform = Affine.translation(float(data.x[0]) - x_res/2, float(data.y[0]) + y_res/2) * Affine.scale(x_res, y_res)
                data = data.rio.write_transform(transform)
            
            # recode for MSPA 
            print(f"Recoding: {filename}")
            recoded = xr.where(data == foreground1, foreground2,
                      xr.where(data == background1, background2, 0)).astype('uint8')
            
            recoded = recoded.rio.write_crs(data.rio.crs)
            recoded = recoded.rio.write_transform(data.rio.transform())
            recoded = recoded.rio.write_nodata(0) 

            # reproject
            if target_crs:
                print(f"Reprojecting: {filename}")
                recoded = recoded.rio.reproject(
                    dst_crs=target_crs,
                    resampling=Resampling.nearest,
                    transform = ##############################
                )
            
            # file naming
            match = re.search(tif_pattern, filename)
            output_name = f"{match.group(1)}_P_recoded.tif" if match else f"{Path(tif_file).stem}_P_recoded.tif"
            output_path = os.path.join(output_dir, output_name)
            
            # export images
            print(f"Exporting: {filename}")
            recoded.rio.to_raster(
                output_path,
                dtype="uint8",
                compress='LZW',
                tiled=True,
                lock=True
            )
            print(f"Saved: {output_name}")
            results.append(True)

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            results.append(False)
    
    return results


In [7]:
# Verify output CRS
import rioxarray as rxr
test_file = "./data/DS_TEST/mosaic_P_recoded/1991_P_recoded.tif"
data = rxr.open_rasterio(test_file)
print("CRS:", data.rio.crs)
print("Is Geographic?", data.rio.crs.is_geographic)
print("Is Projected?", data.rio.crs.is_projected)


CRS: ESRI:102033
Is Geographic? False
Is Projected? True


In [9]:
# Execute the processing
processed = process_files(
    input_dir=input_dir, 
    output_dir=output_dir,
    source_crs=source_crs,
    target_crs=target_crs,
    background1=background1,
    background2=background2,
    foreground1=foreground1,
    foreground2=foreground2,
    tif_pattern=tif_pattern
)
    
# Cleanup
#client.close()
#cluster.close()

Found 1 files to process
Processing: mosaic_1991.tif
Recoding: mosaic_1991.tif
Reprojecting: mosaic_1991.tif


2025-11-17 06:20:05,243 - distributed.worker.memory - WARNING - Worker is at 81% memory usage. Pausing worker.  Process memory: 15.19 GiB -- Worker memory limit: 18.63 GiB
2025-11-17 06:20:07,044 - distributed.worker.memory - WARNING - Worker is at 56% memory usage. Resuming worker. Process memory: 10.44 GiB -- Worker memory limit: 18.63 GiB


Exporting: mosaic_1991.tif
Saved: 1991_P_recoded.tif
